<a href="https://colab.research.google.com/github/humaaslam46/flyRank-ml-Internship-tasks/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/humaaslam46/Internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
# code for file path error

In [5]:
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.isdir("Internship-ml"):
        os.system("git clone --depth 1 https://github.com/humaaslam46/Internship-ml")
    os.chdir("Internship-ml")
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "CSV not found — check you're at repo root"
print("Ready.")

Working dir: /content/Internship-ml/Internship-ml
Ready.


## 1. My rule and its reason codes

**Signal 1 — staleness vs decline rate (behind the refresh flags).** Bucketed
days_since_last_update and checked decline rate per bucket:

| bucket   | n     | decline_rate |
|----------|-------|---------------|
| 0-30     | 20480 | 0.511         |
| 31-90    | 175   | 0.589         |
| 91-180   | 9171  | 0.611         |
| 181-365  | 169   | 0.467         |
| 365+     | 5     | 0.600         |

Verdict: **MIXED**. Decline rate rises from 0-30 to 91-180 days (51.1% -> 61.1%,
both buckets with large n), but drops back to 46.7% at 181-365 days and the
365+ bucket has only 5 rows, too few to trust. Staleness is NOT a clean
monotonic signal past 180 days - so my rule only uses staleness within the
31-180 day window, where the pattern actually holds.

**Signal 2 - CTR vs position tier (behind the CTR-fix logic).** Bucketed by
position_tier:

| tier      | n     | mean_ctr |
|-----------|-------|----------|
| deep      | 1319  | 0.150    |
| page_3_5  | 7242  | 0.223    |
| striking  | 7304  | 0.323    |
| page_1    | 11814 | 0.653    |
| top_3     | 2321  | 1.484    |

Verdict: **CONFIRMED**. Mean CTR rises monotonically as position improves,
across all five tiers, every bucket with a large n (min 1,319). This is a
reliable signal to detect underperformance: a page's CTR relative to its own
tier's average, not CTR in isolation.

**The rule:** score = impressions_90d (visibility) x is_moderately_stale
(31-180 days since update) x (1 / ctr_vs_tier_avg) - so score rises for
pages that are visible, moderately stale, AND underperforming CTR for their
own position tier. Reason code: STALE_AND_CTR_UNDERPERFORM (one code, since
the rule only fires on the combination of both signals). Action label:
review_for_refresh if score > 0, else monitor.

In [6]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

bins = [0, 30, 90, 180, 365, 10000]
labels = ["0-30", "31-90", "91-180", "181-365", "365+"]
df["staleness_bucket"] = pd.cut(df["days_since_last_update"], bins=bins, labels=labels, include_lowest=True)
print("Signal 1: staleness vs decline rate")
print(df.groupby("staleness_bucket", observed=True).agg(n=("is_declining","size"), decline_rate=("is_declining","mean")).round(3))

print("\nSignal 2: position_tier vs mean CTR")
print(df.groupby("position_tier", observed=True).agg(n=("ctr","size"), mean_ctr=("ctr","mean")).round(4).sort_values("mean_ctr"))

Signal 1: staleness vs decline rate
                      n  decline_rate
staleness_bucket                     
0-30              20480         0.511
31-90               175         0.589
91-180             9171         0.611
181-365             169         0.467
365+                  5         0.600

Signal 2: position_tier vs mean CTR
                   n  mean_ctr
position_tier                 
deep            1319    0.1502
page_3_5        7242    0.2225
striking        7304    0.3232
page_1         11814    0.6525
top_3           2321    1.4836


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
import os
tier_avg = df.groupby("position_tier")["ctr"].transform("mean")
df["ctr_vs_tier_avg"] = df["ctr"] / tier_avg.replace(0, 0.0001)
df["is_moderately_stale"] = df["days_since_last_update"].between(31, 180).astype(int)

df["baseline_action_score"] = (
    df["impressions_90d"] * df["is_moderately_stale"] * (1 / df["ctr_vs_tier_avg"].clip(lower=0.05))
)
df["reason_code"] = "STALE_AND_CTR_UNDERPERFORM"
df["action"] = df["baseline_action_score"].apply(lambda s: "review_for_refresh" if s > 0 else "monitor")

queue = df.sort_values("baseline_action_score", ascending=False)

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Wrote {len(queue):,} rows. {(df['action']=='review_for_refresh').sum():,} flagged review_for_refresh.")
queue.head(10)[["content_id","client_id","impressions_90d","days_since_last_update",
                 "position_tier","ctr","ctr_vs_tier_avg","baseline_action_score","action"]]

Wrote 30,000 rows. 9,346 flagged review_for_refresh.


,content_id,client_id,impressions_90d,days_since_last_update,position_tier,ctr,ctr_vs_tier_avg,baseline_action_score,action
7445,content_c8e9d6ab9013,client_19581e27de,208678,104,page_1,0.00,0.000000,4.173560e+06,review_for_refresh
3394,content_36ff89c8214e,client_19581e27de,295097,104,page_1,0.05,0.076632,3.850819e+06,review_for_refresh
9193,content_c1fe78bc4e37,client_19581e27de,134055,104,page_1,0.03,0.045979,2.681100e+06,review_for_refresh
6689,content_e752a4e03dd3,client_6208ef0f77,130892,104,page_3_5,0.01,0.044947,2.617840e+06,review_for_refresh
3343,content_54baba704595,client_6208ef0f77,130617,104,page_3_5,0.01,0.044947,2.612340e+06,review_for_refresh
19183,content_124763d39ca5,client_6208ef0f77,129803,104,page_3_5,0.01,0.044947,2.596060e+06,review_for_refresh
3331,content_4a6607efcb46,client_6208ef0f77,128068,104,top_3,0.01,0.006740,2.561360e+06,review_for_refresh
4708,content_b115f7c74779,client_19581e27de,123469,104,page_1,0.03,0.045979,2.469380e+06,review_for_refresh
6653,content_5fe46e04994d,client_4e07408562,517715,104,page_1,0.14,0.214570,2.412798e+06,review_for_refresh
13693,content_32cfb0b2fccf,client_6208ef0f77,89361,104,page_3_5,0.01,0.044947,1.787220e+06,review_for_refresh


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [10]:
review_notes = {
    "content_c8e9d6ab9013": ("208,678 impressions, page_1, CTR=0.00 vs tier expectation",
                              "if CTR=0.00 is a tracking gap, not a real zero"),
    "content_36ff89c8214e": ("295,097 impressions, CTR far below page_1 average",
                              "if this page just launched and hasn't accrued clicks yet"),
    "content_c1fe78bc4e37": ("134,055 impressions, same client, same pattern",
                              "low CTR could be a measurement issue, not content quality"),
    "content_e752a4e03dd3": ("same client (6208ef0f77), page_3_5, near-identical score to rows below",
                              "three near-duplicate rows from one client may be one real problem counted three times"),
    "content_54baba704595": ("same client, page_3_5, near-identical score",
                              "same client-duplication risk as the row above"),
    "content_124763d39ca5": ("same client, page_3_5, near-identical score",
                              "same client-duplication risk"),
    "content_4a6607efcb46": ("top_3 position but CTR far below top_3's typical average",
                              "worth checking the top_3 tier average isn't itself skewed by outliers"),
    "content_b115f7c74779": ("same client again, page_1",
                              "same client-clustering concern as rows 4-6"),
    "content_5fe46e04994d": ("517,715 impressions (highest in top 10), CTR=0.14, not zero",
                              "least likely of the ten to be a data artifact - most real signal here"),
    "content_32cfb0b2fccf": ("same client cluster, page_3_5",
                              "same duplication concern as rows 4-6 and 8"),
}

top10 = queue.head(10).copy()
top10["why_here"] = top10["content_id"].map(lambda x: review_notes[x][0])
top10["what_would_make_it_wrong"] = top10["content_id"].map(lambda x: review_notes[x][1])

review_table = top10[["content_id", "client_id", "action", "why_here", "what_would_make_it_wrong"]]
review_table

,content_id,client_id,action,why_here,what_would_make_it_wrong
7445,content_c8e9d6ab9013,client_19581e27de,review_for_refresh,"208,678 impressions, page_1, CTR=0.00 vs tier ...","if CTR=0.00 is a tracking gap, not a real zero"
3394,content_36ff89c8214e,client_19581e27de,review_for_refresh,"295,097 impressions, CTR far below page_1 average",if this page just launched and hasn't accrued ...
9193,content_c1fe78bc4e37,client_19581e27de,review_for_refresh,"134,055 impressions, same client, same pattern","low CTR could be a measurement issue, not cont..."
6689,content_e752a4e03dd3,client_6208ef0f77,review_for_refresh,"same client (6208ef0f77), page_3_5, near-ident...",three near-duplicate rows from one client may ...
3343,content_54baba704595,client_6208ef0f77,review_for_refresh,"same client, page_3_5, near-identical score",same client-duplication risk as the row above
19183,content_124763d39ca5,client_6208ef0f77,review_for_refresh,"same client, page_3_5, near-identical score",same client-duplication risk
3331,content_4a6607efcb46,client_6208ef0f77,review_for_refresh,top_3 position but CTR far below top_3's typic...,worth checking the top_3 tier average isn't it...
4708,content_b115f7c74779,client_19581e27de,review_for_refresh,"same client again, page_1",same client-clustering concern as rows 4-6
6653,content_5fe46e04994d,client_4e07408562,review_for_refresh,"517,715 impressions (highest in top 10), CTR=0...",least likely of the ten to be a data artifact ...
13693,content_32cfb0b2fccf,client_6208ef0f77,review_for_refresh,"same client cluster, page_3_5",same duplication concern as rows 4-6 and 8


## 4. Weak picks + leakage check

Weak pattern: 8 of the top 10 rows share days_since_last_update = 104 exactly,
and several share the same client_id with near-identical scores (rows 4-6, 8,
10 all trace to client_6208ef0f77). This looks like a batch effect in the
underlying data generation, not eight independently discovered problems - a
real review queue should probably deduplicate or cap picks per client so one
client's data quirk doesn't fill half the top 10.

Leakage check: no product flags, health scores, or future-window fields were
used - only impressions_90d, days_since_last_update, ctr, and position_tier,
all knowable at the decision moment. trend_direction/trend_pct (the
label-derived fields from earlier notebooks) were not used anywhere in this
score.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.